# 문자 단위 RNN

## 1. 문자 단위 RNN

문자 시퀀스 apple을 입력받으면 pple!을 출력하는 RNN을 구현

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [2]:
input_str = 'apple'
label_str = 'pple!'
char_vocab = sorted(list(set(input_str + label_str)))
vocab_size = len(char_vocab)
print('문자 집합의 크기 : {}'.format(vocab_size))

문자 집합의 크기 : 5


In [3]:
input_size = vocab_size
hidden_size = 5
output_size = 5
learning_rate = 0.1

char_to_index = dict((c, i) for i, c in enumerate(char_vocab))
print(char_to_index)

{'!': 0, 'a': 1, 'e': 2, 'l': 3, 'p': 4}


In [4]:
index_to_char = {}
for key, value in char_to_index.items():
    index_to_char[value] = key
print(index_to_char)

{0: '!', 1: 'a', 2: 'e', 3: 'l', 4: 'p'}


In [5]:
x_data = [char_to_index[c] for c in input_str]
y_data = [char_to_index[c] for c in label_str]
print(x_data) # a, p, p, l, e
print(y_data) # p, p, l, e, !

[1, 4, 4, 3, 2]
[4, 4, 3, 2, 0]


In [6]:
x_data = [x_data]
y_data = [y_data]
print(x_data)
print(y_data)

[[1, 4, 4, 3, 2]]
[[4, 4, 3, 2, 0]]


In [7]:
x_one_hot = [np.eye(vocab_size)[x] for x in x_data] 
#결국 내부적으로 np.eye(5)[[1, 4, 4, 3, 2]] 이런식으로 하게 되는데
#이건 Numpy의 "고급 인덱싱"기능임 그러니까 보고 뭔지 모르는거 정상
#어쨌든 이 코드가 하는 일은 기존의 리스트로 새로운 리스트를 만드는데 그 순서를 지정한 것
print(x_one_hot)

[array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.]])]


In [8]:
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기  {}'.format(Y.shape))

훈련 데이터의 크기 : torch.Size([1, 5, 5])
레이블의 크기  torch.Size([1, 5])


/tmp/ipython-input-2535166115.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  X = torch.FloatTensor(x_one_hot)


In [9]:
class Net(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = torch.nn.Linear(hidden_size, output_size, bias=True)

    def forward(self, x):
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x
    
net = Net(input_size, hidden_size, output_size)
outputs = net(X)
print(outputs.shape) # (1, 5, 5) #(배치 차원, timesteps, 출력의 크기)

torch.Size([1, 5, 5])


In [10]:
print(outputs.view(-1, vocab_size).shape) # (5, 5)

torch.Size([5, 5])


In [11]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([1, 5])
torch.Size([5])


In [12]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=learning_rate)

In [13]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X)
    loss = criterion(outputs.view(-1, input_size), Y.view(-1))
    loss.backward()
    optimizer.step()

    result = outputs.data.numpy().argmax(axis=2)
    # outputs : torch.Tensor
    # outputs.data.numpy() : torch.Tensor -> numpy.ndarray
    # nowdays, .data는 권장되지 않는 방법이므로 .detach()를 사용하는 것이 좋습니다.
    # outputs.detach().numpy() : torch.Tensor -> numpy.ndarray
    # argmax(axis=2) : 3차원 배열에서 마지막 차원(출력의 크기)에서 가장 큰 값의 인덱스를 반환
    result_str = ''.join([index_to_char[c] for c in np.squeeze(result)])
    print('Epoch: {}, Loss: {:.4f} Result: {}'.format(i, loss.item(), result_str))

Epoch: 0, Loss: 1.6946 Result: lllll
Epoch: 1, Loss: 1.4023 Result: pllpl
Epoch: 2, Loss: 1.1978 Result: pppe!
Epoch: 3, Loss: 1.0223 Result: pppe!
Epoch: 4, Loss: 0.8614 Result: pppe!
Epoch: 5, Loss: 0.7173 Result: pppe!
Epoch: 6, Loss: 0.5869 Result: pp!e!
Epoch: 7, Loss: 0.4637 Result: pple!
Epoch: 8, Loss: 0.3579 Result: pple!
Epoch: 9, Loss: 0.2729 Result: pple!
Epoch: 10, Loss: 0.2089 Result: pple!
Epoch: 11, Loss: 0.1644 Result: pple!
Epoch: 12, Loss: 0.1340 Result: pple!
Epoch: 13, Loss: 0.1114 Result: pple!
Epoch: 14, Loss: 0.0927 Result: pple!
Epoch: 15, Loss: 0.0764 Result: pple!
Epoch: 16, Loss: 0.0627 Result: pple!
Epoch: 17, Loss: 0.0515 Result: pple!
Epoch: 18, Loss: 0.0427 Result: pple!
Epoch: 19, Loss: 0.0360 Result: pple!
Epoch: 20, Loss: 0.0308 Result: pple!
Epoch: 21, Loss: 0.0267 Result: pple!
Epoch: 22, Loss: 0.0235 Result: pple!
Epoch: 23, Loss: 0.0208 Result: pple!
Epoch: 24, Loss: 0.0186 Result: pple!
Epoch: 25, Loss: 0.0167 Result: pple!
Epoch: 26, Loss: 0.015

## 2. Char RNN

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim

In [15]:
sentence = ("if you want you build a ship, don't drum up pepole together to "
            "collect wood and don't assgin them task and work, but ratehr "
            "teach them to long for the endless immensity of the sea.")
char_set = list(set(sentence))
char_dic = {c: i for i, c in enumerate(char_set)} #각 문자와 정수 매핑

In [16]:
dic_size = len(char_dic)
print('문자 집합의 크기 : {}'.format(dic_size))

문자 집합의 크기 : 25


In [17]:
hidden_size = 15 # 아무 값이나 넣어도 ㅇㅋ
sequence_length = 10 #문자열을 10개씩 끊어서 RNN에 입력
learning_rate = 0.1

In [18]:
#
x_data = []
y_data = []

for i in range(0, len(sentence) - sequence_length):
    x_str = sentence[i:i+sequence_length]
    y_str = sentence[i+1:i+sequence_length+1]
    print(i, x_str, '->', y_str)

    x_data.append([char_dic[c] for c in x_str])
    y_data.append([char_dic[c] for c in y_str])

0 if you wan -> f you want
1 f you want ->  you want 
2  you want  -> you want y
3 you want y -> ou want yo
4 ou want yo -> u want you
5 u want you ->  want you 
6  want you  -> want you b
7 want you b -> ant you bu
8 ant you bu -> nt you bui
9 nt you bui -> t you buil
10 t you buil ->  you build
11  you build -> you build 
12 you build  -> ou build a
13 ou build a -> u build a 
14 u build a  ->  build a s
15  build a s -> build a sh
16 build a sh -> uild a shi
17 uild a shi -> ild a ship
18 ild a ship -> ld a ship,
19 ld a ship, -> d a ship, 
20 d a ship,  ->  a ship, d
21  a ship, d -> a ship, do
22 a ship, do ->  ship, don
23  ship, don -> ship, don'
24 ship, don' -> hip, don't
25 hip, don't -> ip, don't 
26 ip, don't  -> p, don't d
27 p, don't d -> , don't dr
28 , don't dr ->  don't dru
29  don't dru -> don't drum
30 don't drum -> on't drum 
31 on't drum  -> n't drum u
32 n't drum u -> 't drum up
33 't drum up -> t drum up 
34 t drum up  ->  drum up p
35  drum up p -> drum up pe
36

In [19]:
print(x_data[0])
print(y_data[0])

[22, 18, 17, 13, 7, 11, 17, 9, 0, 10]
[18, 17, 13, 7, 11, 17, 9, 0, 10, 6]


In [20]:
x_one_hot = [np.eye(dic_size)[x] for x in x_data]
X = torch.FloatTensor(x_one_hot)
Y= torch.LongTensor(y_data)

In [21]:
print('훈련 데이터의 크기 : {}'.format(X.shape)) # 샘플 수 , 시퀀스 길이, One-hot 벡터의 크기
print('레이블의 크기  {}'.format(Y.shape)) # 샘플 수, 시퀀스 길이 one-hot은 하지 않았으므로 2차원

훈련 데이터의 크기 : torch.Size([170, 10, 25])
레이블의 크기  torch.Size([170, 10])


In [22]:
print(X[0])
print(Y[0])

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,

In [23]:
class Net(torch.nn.Module):
    def __init__(self, input_size, hidden_size, layers):
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_size, hidden_size, batch_first=True, num_layers=layers)
        self.fc = torch.nn.Linear(hidden_size, input_size, bias=True)

    def forward(self, x):
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x
    
net = Net(dic_size, hidden_size, 2) #(선형 + 활성화) + (선형 + 활성화) + 선형

In [24]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=learning_rate)

outputs = net(X)
print(outputs.shape)

torch.Size([170, 10, 25])


In [25]:
print(outputs.view(-1, dic_size).shape)
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([1700, 25])
torch.Size([170, 10])
torch.Size([1700])


In [ ]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X)
    loss = criterion(outputs.view(-1, dic_size), Y.view(-1))
    loss.backward()
    optimizer.step()

    result = outputs.argmax(dim=2) #one hot vector -> index
    predict_str = ""
    for j, result in enumerate(result): #enumerate안에 2차원 이상의 배열을 넣으면 가장 바깥 껍데기만 벗김
        if j == 0:
            predict_str += ''.join([char_set[t] for t in result])
        else:
            predict_str += char_set[result[-1]]

    print(predict_str)

tensor([20, 17, 13,  7, 11, 17,  9,  0, 10,  6])
tensor([ 7, 13,  7, 11, 17,  9,  0, 10,  6, 17])
tensor([ 6,  7, 11, 17, 24,  0, 10,  6, 17, 13])
tensor([ 7, 11, 17, 24,  0, 10,  6, 17, 13,  7])
tensor([10, 17, 24,  0, 10,  6, 17, 13,  7, 11])
tensor([17, 24,  0, 10,  6, 17, 13,  7, 11, 17])
tensor([ 6,  7, 10,  6, 17, 13,  7, 11, 17, 24])
tensor([ 7, 10,  6, 17, 13,  7, 11, 17, 24, 11])
tensor([10, 23, 17, 13,  7, 11, 17, 24, 11, 22])
tensor([23, 17, 13,  7, 11, 17, 24, 11, 22,  3])
tensor([12,  1,  7, 11, 17, 24, 11, 22,  3, 23])
tensor([ 6,  7, 11, 17, 24, 11, 22,  3, 23, 17])
tensor([ 7, 11, 17, 24, 11, 22,  3, 23, 17,  0])
tensor([10, 17, 24, 11, 22,  3, 23, 17,  0, 17])
tensor([17, 24, 11, 22,  3, 23, 17,  0, 17, 16])
tensor([ 6, 11, 22,  3, 23, 17,  0, 17, 16, 12])
tensor([11, 22,  3, 23, 17,  0, 17, 16, 12, 22])
tensor([17,  3, 23, 17,  0, 17, 16, 12, 22, 20])
tensor([20, 23, 17,  0, 17, 16, 12, 22, 20, 15])
tensor([ 2, 17,  0, 17, 16, 12, 22, 20, 15, 17])
tensor([17,  0, 10, 